In [20]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from pprint import pprint
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tabulate import tabulate
import asyncio
import nest_asyncio

nest_asyncio.apply()

import os
import random

config = {
    "dataset":{
        "dti":"../../Data/scope_onside_common_v3.parquet",
        "adr":"../../Data/final_rxnorm_meddra_v2.parquet"
    },
    "protein_emb_1":{
        "path":  "../../Data/3. Protein_enbeddings/ESM_embeddings_(t33_650m model).parquet",
        "id_col": "id", 
        "emb_col": "embedding"
    },
    "protein_emb_2":{
        "path": "../../Data/3. Protein_enbeddings/GVP-GNN_protein_embeddings.parquet",
        "id_col": "uniprot_id", 
        "emb_col": "embedding"
    },
    "drug_emb_1":{
        "path": "../../Data/2. Drug_embeddings/EGNN_drug_embeddings_v2.parquet", 
        "id_col": "drug_chembl_id", 
        "emb_col": "embedding"
    },
    "drug_emb_2":{
        "path": "../../Data/2. Drug_embeddings/smiles_embeddings_chemberta.parquet", 
        "id_col": "drug_chembl_id", 
        "emb_col": "embedding"
    }
}

In [21]:
dti_df = pd.read_parquet(config["dataset"]["dti"])
print(dti_df.info())

# copy selected columns to a new df
# drug_chembl_id as drug_id and target_uniprot_id as protein_id
dti_df = dti_df.rename(columns={"drug_chembl_id": "drug_id", "target_uniprot_id": "protein_id"})

df = dti_df.copy()
df = df[["drug_id", "protein_id", "label", "rxcui"]]


if config["protein_emb_1"]["path"]:
    protein_emb_1_df = pd.read_parquet(config["protein_emb_1"]["path"])
    protein_emb_1_df = protein_emb_1_df.rename(columns={config["protein_emb_1"]["id_col"]: "protein_id"})
    df = df.merge(protein_emb_1_df[["protein_id", config["protein_emb_1"]["emb_col"]]], on="protein_id", how="left")
    df = df.rename(columns={config["protein_emb_1"]["emb_col"]: "prot_emb_1"})

if config["protein_emb_2"]["path"]:
    protein_emb_2_df = pd.read_parquet(config["protein_emb_2"]["path"])
    protein_emb_2_df = protein_emb_2_df.rename(columns={config["protein_emb_2"]["id_col"]: "protein_id"})
    df = df.merge(protein_emb_2_df[["protein_id", config["protein_emb_2"]["emb_col"]]], on="protein_id", how="left")
    df = df.rename(columns={config["protein_emb_2"]["emb_col"]: "prot_emb_2"})

if config["drug_emb_1"]["path"]:
    drug_emb_1_df = pd.read_parquet(config["drug_emb_1"]["path"])
    drug_emb_1_df = drug_emb_1_df.rename(columns={config["drug_emb_1"]["id_col"]: "drug_id"})
    df = df.merge(drug_emb_1_df[["drug_id", config["drug_emb_1"]["emb_col"]]], on="drug_id", how="left")
    df = df.rename(columns={config["drug_emb_1"]["emb_col"]: "drug_emb_1"})

if config["drug_emb_2"]["path"]:
    drug_emb_2_df = pd.read_parquet(config["drug_emb_2"]["path"])
    drug_emb_2_df = drug_emb_2_df.rename(columns={config["drug_emb_2"]["id_col"]: "drug_id"})
    df = df.merge(drug_emb_2_df[["drug_id", config["drug_emb_2"]["emb_col"]]], on="drug_id", how="left")
    df = df.rename(columns={config["drug_emb_2"]["emb_col"]: "drug_emb_2"})



print(df.info())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34741 entries, 0 to 34740
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   drug_chembl_id     34741 non-null  object
 1   target_uniprot_id  34741 non-null  object
 2   label              34741 non-null  int64 
 3   smiles             34741 non-null  object
 4   sequence           34741 non-null  object
 5   molfile_3d         34741 non-null  object
 6   rxcui              34741 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.9+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34741 entries, 0 to 34740
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   drug_id     34741 non-null  object
 1   protein_id  34741 non-null  object
 2   label       34741 non-null  int64 
 3   rxcui       34741 non-null  object
 4   prot_emb_1  34741 non-null  object
 5   prot_emb_2  34741 non

In [22]:

class ADRData:
    def __init__(self, id_to_name_dict):
        """
        Initializes with a dictionary of {meddra_id: meddra_name}.
        """
        self.id_to_name = id_to_name_dict
        self.unique_ids = sorted(list(id_to_name_dict.keys()))
        
        self.id_to_idx = {adr_id: i for i, adr_id in enumerate(self.unique_ids)}
        self.idx_to_id = {i: adr_id for i, adr_id in enumerate(self.unique_ids)}
        
        self.vocab_size = len(self.unique_ids)

    def encode(self, adr_list):
        """
        Takes a list of ADR IDs and returns a binary vector (1s and 0s).
        Example: ['10028553', '10003041'] -> [0, 1, 0, 0, 1...]
        """
        vector = np.zeros(self.vocab_size, dtype=np.int8)
        
        for adr_id in adr_list:
            if adr_id in self.id_to_idx:
                idx = self.id_to_idx[adr_id]
                vector[idx] = 1
            else:
                print(f"Warning: ADR ID {adr_id} not in vocabulary.")
                
        return vector

    def decode(self, vector):
        """
        Takes a binary vector and returns a list of human-readable ADR names.
        """
        decoded_names = []
        
        active_indices = np.where(vector == 1)[0]
        
        for idx in active_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            decoded_names.append(name)
            
        return decoded_names

    def decode_indices(self, indices):
        """
        Takes a list or array of indices (e.g., [42, 105, 300]) 
        and returns the corresponding ADR names.
        """
        return [self.id_to_name.get(self.idx_to_id[idx], "Unknown ADR") for idx in indices]

    def decode_top_k(self, confidence_array, k=5):
        """
        Takes the raw probability array from the model, finds the top K 
        highest values, and returns names + their confidence scores.
        """
        # Get indices of the top k probabilities
        top_indices = np.argsort(confidence_array)[-k:][::-1]
        
        results = []
        for idx in top_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            conf = confidence_array[idx]
            results.append({"name": name, "confidence": round(float(conf), 4)})
            
        return results

    def decode_with_threshold(self, confidence_array, threshold=0.5):
        """
        Returns all ADRs that pass a specific confidence threshold.
        Useful for seeing everything the model is "sure" about.
        """
        active_indices = np.where(confidence_array >= threshold)[0]
        
        # Sort them by confidence (highest first)
        active_indices = active_indices[np.argsort(confidence_array[active_indices])[::-1]]
        
        return [self.id_to_name.get(self.idx_to_id[idx], "Unknown ADR") for idx in active_indices]


In [23]:
adrdf = pd.read_parquet(config['dataset']["adr"])
id_name_dict = dict(zip(adrdf['meddra_id'], adrdf['meddra_name']))
adr_manager = ADRData(id_name_dict)

In [24]:
drug_to_adr_list = adrdf.groupby('rxnorm_ingredient_id')['meddra_id'].apply(list).to_dict()

def get_encoded_adr(drug_id):
    # Get the list of ADRs for this drug, or an empty list if not found
    adrs = drug_to_adr_list.get(drug_id, [])
    return adr_manager.encode(adrs)

# 2. Map the drug_id (e.g., rxcui) to the encoded vector
# This will create a column where each cell is a numpy array
df['adr'] = df['rxcui'].map(get_encoded_adr)
print(f"Total rows with ADRs: {df['adr'].apply(lambda x: x.sum() > 0).sum()}")

Total rows with ADRs: 34741


In [25]:
# print number of unique protein and drug ids

print(df["protein_id"].nunique())
print(df["drug_id"].nunique())

# print number of unique rxcui
print(df["rxcui"].nunique())
print(df.head(1))



2385
1028
1028
      drug_id protein_id  label  rxcui  \
0  CHEMBL1000     O15245      0  20610   

                                          prot_emb_1  \
0  [-0.041778564453125, 0.0305938720703125, -0.01...   

                                          prot_emb_2  \
0  [0.143310546875, 0.340087890625, -0.3349609375...   

                                          drug_emb_1  \
0  [0.02189382165670395, 0.016782937571406364, -0...   

                                          drug_emb_2  \
0  [0.008678080514073372, -0.08597195148468018, -...   

                                                 adr  
0  [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, ...  


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34741 entries, 0 to 34740
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   drug_id     34741 non-null  object
 1   protein_id  34741 non-null  object
 2   label       34741 non-null  int64 
 3   rxcui       34741 non-null  object
 4   prot_emb_1  34741 non-null  object
 5   prot_emb_2  34741 non-null  object
 6   drug_emb_1  34741 non-null  object
 7   drug_emb_2  34741 non-null  object
 8   adr         34741 non-null  object
dtypes: int64(1), object(8)
memory usage: 2.4+ MB


In [27]:
from sklearn.model_selection import train_test_split

# 1. Get all unique protein IDs
unique_proteins = df['protein_id'].unique()

# 2. Split protein IDs (not rows) to ensure no leakage
# We'll reserve 10% of proteins for Test and 10% for Validation
train_prot_ids, temp_prot_ids = train_test_split(
    unique_proteins, 
    test_size=0.20, 
    random_state=42
)

val_prot_ids, test_prot_ids = train_test_split(
    temp_prot_ids, 
    test_size=0.50, 
    random_state=42
)

# 3. Create the dataframes based on these ID splits
train_df = df[df['protein_id'].isin(train_prot_ids)]
val_df = df[df['protein_id'].isin(val_prot_ids)]
test_df = df[df['protein_id'].isin(test_prot_ids)]

# --- Verification & Metrics ---
print(f"--- Final Dataset Sizes ---")
print(f"Train Set: {len(train_df)} rows ({len(train_prot_ids)} proteins)")
print(f"Val Set:   {len(val_df)} rows ({len(val_prot_ids)} proteins) - [Model Selection]")
print(f"Test Set:  {len(test_df)} rows ({len(test_prot_ids)} proteins) - [Cold-Protein Eval]")

# 4. Recalculate Positive Weight for Training
num_neg = (train_df['label'] == 0).sum()
num_pos = (train_df['label'] == 1).sum()

# Avoid division by zero just in case
pos_weight_value = num_neg / num_pos if num_pos > 0 else 1.0

print(f"\nNew Positive Weight: {pos_weight_value:.2f}")
print(f"Positive/Negative Ratio in Train: 1:{num_neg/num_pos:.2f}")

--- Final Dataset Sizes ---
Train Set: 28055 rows (1908 proteins)
Val Set:   3335 rows (238 proteins) - [Model Selection]
Test Set:  3351 rows (239 proteins) - [Cold-Protein Eval]

New Positive Weight: 1.75
Positive/Negative Ratio in Train: 1:1.75


In [33]:
# =========================
# DTI BASELINE: LINEAR SVM (SGDClassifier) with tqdm + live Val AUPRC
# =========================
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score

def _stack_col(df_in, col_name):
    """Stack an embedding column (each row is list/np.array) into a 2D float32 array with tqdm."""
    arrs = []
    for a in tqdm(df_in[col_name].values, desc=f"Stacking {col_name}", leave=False):
        arrs.append(np.asarray(a, dtype=np.float32))
    return np.stack(arrs, axis=0)

def build_dti_Xy(df_in):
    """
    Build X and y for DTI.
    Uses your embedding columns:
      X = concat([drug_emb_1, drug_emb_2, prot_emb_1, prot_emb_2])
      y = label
    """
    X = np.concatenate([
        _stack_col(df_in, "drug_emb_1"),
        _stack_col(df_in, "drug_emb_2"),
        _stack_col(df_in, "prot_emb_1"),
        _stack_col(df_in, "prot_emb_2"),
    ], axis=1)

    y = df_in["label"].to_numpy(dtype=np.int64)
    return X, y

def auprc_from_scores(y_true, scores):
    """AUPRC computed from raw decision scores (works fine for ranking-based metrics)."""
    prec, rec, _ = precision_recall_curve(y_true, scores)
    return auc(rec, prec)

print("\n==============================")
print("Running DTI Linear SVM baseline (SGDClassifier hinge) ...")
print("==============================")

# --- Build features from SAME split (cold-protein) ---
X_train, y_train = build_dti_Xy(train_df)
X_val, y_val     = build_dti_Xy(val_df)
X_test, y_test   = build_dti_Xy(test_df)

# --- Standardize features (important for SGD / SVM) ---
scaler = StandardScaler(with_mean=True, with_std=True)
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# --- Linear SVM via SGD (hinge loss) ---
# We do our own epoch loop using partial_fit so tqdm is real (not fake).
clf = SGDClassifier(
    loss="hinge",            # linear SVM objective
    penalty="l2",
    alpha=1e-4,              # regularization strength
    learning_rate="optimal",
    random_state=42,
    warm_start=True
)

epochs = 500
classes = np.array([0, 1], dtype=np.int64)

best_val_auprc = -1.0
best_state = None  # (coef_, intercept_)

from sklearn.utils.class_weight import compute_class_weight

# Compute balanced class weights using the full training labels
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
cw_map = {cls: w for cls, w in zip(classes, cw)}

# Convert to per-sample weights (SGD partial_fit supports sample_weight)
sample_w_train = np.array([cw_map[int(lbl)] for lbl in y_train], dtype=np.float32)

# --- Early stopping config ---
patience = 50          # stop after N epochs with no improvement
min_delta = 1e-4      # required improvement to count as "better"

best_epoch = 0
bad_epochs = 0

print("Class weights:", cw_map)
print("Sample weight stats:", float(sample_w_train.min()), float(sample_w_train.max()))

pbar = tqdm(range(1, epochs + 1), desc="SGD-SVM epochs", total=epochs)
for ep in pbar:
    clf.partial_fit(X_train, y_train, classes=classes, sample_weight=sample_w_train)

    # raw margins (scores)
    val_scores = clf.decision_function(X_val)
    val_auprc  = auprc_from_scores(y_val, val_scores)

    try:
        val_auroc = roc_auc_score(y_val, val_scores)
    except Exception:
        val_auroc = float("nan")

    improved = val_auprc > (best_val_auprc + min_delta)
    if improved:
        best_val_auprc = val_auprc
        best_epoch = ep
        bad_epochs = 0
        best_state = (clf.coef_.copy(), clf.intercept_.copy())
    else:
        bad_epochs += 1

    pbar.set_postfix({
        "val_auprc": f"{val_auprc:.4f}",
        "val_auroc": f"{val_auroc:.4f}",
        "best": f"{best_val_auprc:.4f}",
        "bad": f"{bad_epochs}/{patience}"
    })

    if bad_epochs >= patience:
        print(f"\nEarly stopping at epoch {ep} (best epoch: {best_epoch}, best Val AUPRC: {best_val_auprc:.4f})")
        break

# Restore best epoch weights (by Val AUPRC) for final reporting
if best_state is not None:
    clf.coef_, clf.intercept_ = best_state
    
print(f"Restored best model from epoch {best_epoch} with Val AUPRC={best_val_auprc:.4f}")

# --- Final metrics ---
val_scores  = clf.decision_function(X_val)
test_scores = clf.decision_function(X_test)

val_auprc  = auprc_from_scores(y_val, val_scores)
test_auprc = auprc_from_scores(y_test, test_scores)

try:
    val_auroc  = roc_auc_score(y_val, val_scores)
    test_auroc = roc_auc_score(y_test, test_scores)
except Exception:
    val_auroc, test_auroc = float("nan"), float("nan")

print("\n[DTI Linear SVM (SGDClassifier hinge) Results]")
print(f"Val  AUROC={val_auroc:.4f} | AUPRC={val_auprc:.4f}")
print(f"Test AUROC={test_auroc:.4f} | AUPRC={test_auprc:.4f}")



Running DTI Linear SVM baseline (SGDClassifier hinge) ...


Stacking drug_emb_1:   0%|          | 0/28055 [00:00<?, ?it/s]

Stacking drug_emb_2:   0%|          | 0/28055 [00:00<?, ?it/s]

Stacking prot_emb_1:   0%|          | 0/28055 [00:00<?, ?it/s]

Stacking prot_emb_2:   0%|          | 0/28055 [00:00<?, ?it/s]

...

Stacking prot_emb_1:   0%|          | 0/3351 [00:00<?, ?it/s]

Stacking prot_emb_2:   0%|          | 0/3351 [00:00<?, ?it/s]

Class weights: {np.int64(0): np.float64(0.7861185832772921), np.int64(1): np.float64(1.3737635882871413)}
Sample weight stats: 0.7861185669898987 1.3737635612487793


SGD-SVM epochs:   0%|          | 0/500 [00:00<?, ?it/s]


Early stopping at epoch 73 (best epoch: 23, best Val AUPRC: 0.6354)
Restored best model from epoch 23 with Val AUPRC=0.6354

[DTI Linear SVM (SGDClassifier hinge) Results]
Val  AUROC=0.7956 | AUPRC=0.6354
Test AUROC=0.8302 | AUPRC=0.7329
